# YouTube Toxic Comment Classification

## Part 1 — Introduction

### Student details

- IdoM — ID ending: [8349]
- ShirZ — ID ending: [4811]
- RoeeS — ID ending: [5498]

### AI prompts and additional resources

| Tool or resource | Prompt | Purpose |
|---|---|---|
| ChatGPT / Codex | "Take a look at the assignment and explain what is required." | Understanding the assignment requirements. |
| ChatGPT / Codex | "Which algorithm options are suitable for this classification problem?" | Selecting a learning algorithm. |
| ChatGPT / Codex | "Do we need to define quality metrics?" | Understanding the required evaluation metric. |
| Kaggle | https://www.kaggle.com/datasets/reihanenamdari/youtube-toxicity-data | Dataset source. |

### Learning problem and dataset

This project addresses a supervised binary text-classification problem: predicting whether an English YouTube comment is toxic. The input is the comment from the `Text` column, and the target is `IsToxic`, where `TRUE` represents a toxic comment and `FALSE` represents a non-toxic comment. The selected Kaggle dataset contains 1,000 manually labelled YouTube comments and additional labels describing different toxicity categories.

In [ ]:
import pandas as pd

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
train_df.head(5)

In [ ]:
test_df.head(5)

# Quality metric

This is a binary classification problem with one central class: toxic comments. Therefore, according to the assignment instructions, model quality will be evaluated using the F1 score for the toxic class only (`IsToxic = 1`).

The F1 score combines precision and recall. Precision measures how many of the comments predicted as toxic are actually toxic, while recall measures how many of the truly toxic comments were correctly identified. This is appropriate for the current task because both failing to detect a toxic comment and incorrectly flagging a non-toxic comment are important errors.

The F1 score is calculated as:

$$
F1 = 2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}
$$

The same metric will be used throughout cross-validation, hyperparameter selection, and final test-set evaluation.

In [ ]:
from sklearn.metrics import f1_score

def calculate_quality(y_true, y_pred):
    """Calculate the F1 score for the toxic class."""
    return f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

# Part 2 — Feature Engineering

## Text preprocessing

Machine-learning algorithms cannot process raw text directly. Therefore, the comments must first be normalized and tokenized.

The preprocessing includes conversion to lowercase, removal of HTML tags and URLs, and tokenization. Stop-word removal and Porter stemming are optional operations whose effect will later be evaluated using 5-fold cross-validation. Negation words such as `not`, `no`, and `never` are preserved because removing them may change the meaning of a sentence.

The original comments remain unchanged. Only the text supplied to the feature-extraction stage is transformed.

In [ ]:
import html
import re
import numpy as np

from nltk.stem import PorterStemmer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

porter_stemmer = PorterStemmer()
negation_words = {"no", "nor", "not", "never"}
safe_stop_words = set(ENGLISH_STOP_WORDS) - negation_words


class TextPreprocessor(BaseEstimator, TransformerMixin):
    """Clean and normalize English comments."""

    def __init__(self, remove_stopwords=False, use_stemming=False):
        self.remove_stopwords = remove_stopwords
        self.use_stemming = use_stemming

    def fit(self, X, y=None):
        return self

    def clean_comment(self, text):
        text = html.unescape(str(text)).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"\bcan't\b", "can not", text)
        text = re.sub(r"\bwon't\b", "will not", text)
        text = re.sub(r"n't\b", " not", text)
        tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in safe_stop_words]

        if self.use_stemming:
            tokens = [porter_stemmer.stem(token) for token in tokens]

        return " ".join(tokens)

    def transform(self, X):
        return [self.clean_comment(text) for text in X]

The preprocessing procedure is demonstrated on three comments from the training set and three comments from the test set. For each comment, basic preprocessing, stop-word removal, and Porter stemming are presented. The test examples are transformed only for demonstration and are not used to learn a vocabulary or select preprocessing parameters.

In [ ]:
basic_preprocessor = TextPreprocessor(
    remove_stopwords=False,
    use_stemming=False,
)

stopword_preprocessor = TextPreprocessor(
    remove_stopwords=True,
    use_stemming=False,
)

stemming_preprocessor = TextPreprocessor(
    remove_stopwords=True,
    use_stemming=True,
)

selected_train_indices = train_df.sample(n=3, random_state=42).index.tolist()
selected_test_indices = test_df.sample(n=3, random_state=42).index.tolist()

### Three preprocessing examples from the train and test sets

In [ ]:
def create_preprocessing_examples(source_df, row_indices, dataset_name):
    examples = source_df.loc[row_indices, ["Text", "IsToxic"]].copy()
    examples.insert(0, "Dataset", dataset_name)
    examples["Basic preprocessing"] = basic_preprocessor.transform(examples["Text"])
    examples["Without stop words"] = stopword_preprocessor.transform(examples["Text"])
    examples["With Porter stemming"] = stemming_preprocessor.transform(examples["Text"])
    return examples.reset_index(drop=True)


train_preprocessing_examples = create_preprocessing_examples(
    train_df, selected_train_indices, "Train"
)
test_preprocessing_examples = create_preprocessing_examples(
    test_df, selected_test_indices, "Test"
)

display(pd.concat(
    [train_preprocessing_examples, test_preprocessing_examples],
    ignore_index=True,
))

## TF-IDF feature extraction

After preprocessing, the comments are converted into numerical feature vectors using TF-IDF. Term Frequency measures how frequently a term occurs in a particular comment. Inverse Document Frequency reduces the weight of terms that occur in many training comments and increases the relative importance of less common terms.

$$\operatorname{TFIDF}(t,d)=\operatorname{TF}(t,d)\cdot\operatorname{IDF}(t)$$

The representation includes unigrams and bigrams. Unigrams represent individual words, while bigrams represent pairs of adjacent words and preserve limited local context. Sublinear term frequency is applied so repeated appearances of a term do not increase its influence linearly. L2 normalization scales every comment vector to unit length.

The vectorizer is fitted on the training set only. The test set is transformed using the vocabulary and IDF values learned from the training set.

In [ ]:
processed_train_text = basic_preprocessor.transform(train_df["Text"])
processed_test_text = basic_preprocessor.transform(test_df["Text"])

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2",
)

X_train_tfidf = tfidf_vectorizer.fit_transform(processed_train_text)
X_test_tfidf = tfidf_vectorizer.transform(processed_test_text)

print("Train feature matrix shape:", X_train_tfidf.shape)
print("Test feature matrix shape:", X_test_tfidf.shape)

### TF-IDF examples

For each example, the following function displays the original comment and the eight features with the highest non-zero TF-IDF weights. Each feature represents either one word or a pair of adjacent words.

In [ ]:
def show_tfidf_examples(source_df, feature_matrix, vectorizer, row_indices, dataset_name, top_n=8):
    feature_names = vectorizer.get_feature_names_out()
    result_rows = []

    for row_index in row_indices:
        feature_row = feature_matrix.getrow(row_index)
        sorted_positions = np.argsort(feature_row.data)[::-1][:top_n]
        highest_features = []

        for position in sorted_positions:
            feature_index = feature_row.indices[position]
            feature_name = feature_names[feature_index]
            feature_value = feature_row.data[position]
            highest_features.append(f"{feature_name}: {feature_value:.3f}")

        result_rows.append({
            "Dataset": dataset_name,
            "IsToxic": source_df.loc[row_index, "IsToxic"],
            "Text": source_df.loc[row_index, "Text"],
            "Highest TF-IDF features": ", ".join(highest_features),
        })

    return pd.DataFrame(result_rows)

### Three train-set examples

In [ ]:
train_tfidf_examples = show_tfidf_examples(
    train_df,
    X_train_tfidf,
    tfidf_vectorizer,
    selected_train_indices,
    "Train",
)
display(train_tfidf_examples)

### Three test-set examples

In [ ]:
test_tfidf_examples = show_tfidf_examples(
    test_df,
    X_test_tfidf,
    tfidf_vectorizer,
    selected_test_indices,
    "Test",
)
display(test_tfidf_examples)